# Network Security System — Phishing Detection Pipeline

Study notebook: walks through every pipeline stage interactively, using the
same `networksecurity/` package the real pipeline (`main.py`, `app.py`,
Docker, GitHub Actions) imports — so this notebook and the production code
never diverge.

**Dataset:** UCI "Phishing Websites" (id 327) — 11,055 records, 30 binary/
ternary features describing a website's URL structure and page behavior,
target `result` in {-1 (phishing), 1 (legitimate)}.

**Pipeline:** CSV -> MongoDB (ETL) -> Ingestion -> Validation (schema + drift)
-> Transformation (KNNImputer) -> Model Trainer (6 algorithms x
GridSearchCV, tracked in MLflow) -> saved model -> FastAPI / Docker / GitHub
Actions CI-CD -> EC2.


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv()


## EDA: pulling the raw dataset

Using `ucimlrepo` to fetch directly from the UCI ML Repository (id=327) —
reproducible, no manual download/guessed URL needed.


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

ds = fetch_ucirepo(id=327)
X, y = ds.data.features, ds.data.targets
df = pd.concat([X, y], axis=1)

print(df.shape)
df.head()


In [ ]:
df['result'].value_counts()
# 1 = legitimate, -1 = phishing. Fairly balanced (~56%/44%), so accuracy
# alone wouldn't be misleading here, but we still track F1/precision/recall.


In [ ]:
df.isna().sum().sum()  # 0 — this dataset has no missing values out of the box;
# KNNImputer in the transformation stage is defensive/general-purpose, not
# strictly required for THIS dataset, but keeps the pipeline correct if the
# data source ever changes.


## ETL: load into MongoDB

`push_data.py` is the standalone script that does this (run once, not part
of every training run). Shown here inline for the notebook walkthrough —
the real entry point is the script itself.


In [ ]:
from push_data import NetworkDataExtract
from networksecurity.constants.training_pipeline import DATA_INGESTION_COLLECTION_NAME, DATA_INGESTION_DATABASE_NAME

extractor = NetworkDataExtract()
records = extractor.csv_to_json_converter(os.path.join('Network_Data', 'phisingData.csv'))
count = extractor.insert_data_mongodb(records, DATA_INGESTION_DATABASE_NAME, DATA_INGESTION_COLLECTION_NAME)
print(f'{count} records in MongoDB')


## Stage 1 — Data Ingestion

Pulls everything back OUT of Mongo, snapshots it to a feature-store CSV, and
does the train/test split (80/20). Ingesting from Mongo rather than the CSV
directly means swapping the data source later never touches pipeline code.


In [ ]:
from networksecurity.components.data_ingestion import DataIngestion
from networksecurity.entity.config_entity import DataIngestionConfig, TrainingPipelineConfig

pipeline_config = TrainingPipelineConfig()
ingestion_config = DataIngestionConfig(pipeline_config)
ingestion = DataIngestion(ingestion_config, mongo_db_url=os.getenv('MONGO_DB_URL'))
ingestion_artifact = ingestion.initiate_data_ingestion()
ingestion_artifact


## Stage 2 — Data Validation

Two checks: column-count matches `data_schema/schema.yaml`, and a per-column
Kolmogorov-Smirnov test between train/test distributions (dataset drift). A
low p-value means train and test look like they're drawn from different
distributions — would make test metrics misleading.


In [ ]:
from networksecurity.components.data_validation import DataValidation
from networksecurity.entity.config_entity import DataValidationConfig

validation_config = DataValidationConfig(pipeline_config)
validation = DataValidation(ingestion_artifact, validation_config)
validation_artifact = validation.initiate_data_validation()
print('validation_status:', validation_artifact.validation_status)


## Stage 3 — Data Transformation

`KNNImputer` (k=3) over `SimpleImputer`/mean: these phishing-indicator
features are correlated with each other, so a similar row's values are a
better guess for a missing value than the column mean. Output is saved as
numpy arrays (`.npy`), not CSV — the model trainer wants arrays, not a
DataFrame, and `.npy` round-trips exactly without dtype surprises.


In [ ]:
from networksecurity.components.data_transformation import DataTransformation
from networksecurity.entity.config_entity import DataTransformationConfig

transformation_config = DataTransformationConfig(pipeline_config)
transformation = DataTransformation(validation_artifact, transformation_config)
transformation_artifact = transformation.initiate_data_transformation()
transformation_artifact


## Stage 4 — Model Trainer

6 candidate algorithms (Logistic Regression, Decision Tree, Random Forest,
Gradient Boosting, AdaBoost, XGBoost), each hyperparameter-tuned with
`GridSearchCV(cv=3)`, then compared on held-out test F1 — "Model Trainer And
Evaluation With Hyperparameter Tuning" means a real per-algorithm grid
search and an apples-to-apples comparison, not one `.fit()` call. Every
run's params/metrics/model are logged to MLflow (local `./mlruns` by
default, or a DagsHub remote if `MLFLOW_TRACKING_URI` is set).


In [ ]:
from networksecurity.components.model_trainer import ModelTrainer
from networksecurity.entity.config_entity import ModelTrainerConfig

trainer_config = ModelTrainerConfig(pipeline_config)
trainer = ModelTrainer(trainer_config, transformation_artifact)
trainer_artifact = trainer.initiate_model_trainer()

print('Train F1:', trainer_artifact.train_metric_artifact.f1_score)
print('Test F1:', trainer_artifact.test_metric_artifact.f1_score)


## Inference: NetworkModel

The saved `model.pkl` is a `NetworkModel` — preprocessor + classifier
bundled into one object with a single `.predict()`, so inference code can
never accidentally skip preprocessing or mismatch model/preprocessor
versions.


In [ ]:
from networksecurity.utils.main_utils.utils import load_object

network_model = load_object(os.path.join('saved_models', 'model.pkl'))

sample = pd.read_csv(validation_artifact.valid_test_file_path).drop(columns=['result']).head(5)
preds = network_model.predict(sample)
['legitimate' if p == 1 else 'phishing' for p in preds]


## Where this goes next

`main.py` runs this exact sequence end to end (`TrainingPipeline().run_pipeline()`).
`app.py` (FastAPI) serves `/train` and `/predict` over HTTP. The
`Dockerfile` containerizes `app.py`, and `.github/workflows/main.yml`
builds/pushes that image to ECR and deploys it to EC2 via a self-hosted
runner on every push to `main`. See `README.md` for the full picture and
setup instructions.
